# AML Genomics & Clinical Data Explorer
## Part 1 — Data quality and mutation landscape

This reproducible notebook analyzes open-access, de-identified TCGA-LAML PanCancer Atlas data obtained through cBioPortal. It focuses on cohort integrity, missingness, mutation frequency, and variant classes.

**Responsible use:** No attempt is made to identify participants. Raw patient-level files are intentionally excluded from the public repository. Results are exploratory and are not suitable for diagnosis, prognosis, or treatment decisions.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
DATA_DIR = Path("data/raw")
FIGURE_DIR = Path("reports/figures")
TABLE_DIR = Path("reports/tables")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the three source tables

The parser ignores cBioPortal metadata rows beginning with `#` and reads the remaining tab-separated table.

In [ ]:
patient = pd.read_csv(DATA_DIR / "data_clinical_patient.txt", sep="\t", comment="#", low_memory=False)
sample = pd.read_csv(DATA_DIR / "data_clinical_sample.txt", sep="\t", comment="#", low_memory=False)
mutations = pd.read_csv(DATA_DIR / "data_mutations.txt", sep="\t", comment="#", low_memory=False)
somatic_mutations = mutations.loc[mutations["Mutation_Status"].eq("Somatic")].copy()

cohort_summary = pd.DataFrame({
    "table": ["Clinical patient", "Clinical sample", "Somatic mutations"],
    "rows": [len(patient), len(sample), len(mutations)],
    "columns": [patient.shape[1], sample.shape[1], mutations.shape[1]],
})
cohort_summary

## 2. Integrity checks

These checks verify unique cohort keys and confirm that every mutation record can be linked to the clinical sample table. Only aggregate results are displayed.

In [ ]:
integrity = pd.Series({
    "Unique patients": patient["PATIENT_ID"].nunique(),
    "Unique samples": sample["SAMPLE_ID"].nunique(),
    "Duplicate patient rows": patient.duplicated().sum(),
    "Duplicate sample rows": sample.duplicated().sum(),
    "All mutation-table rows": len(mutations),
    "Validated somatic mutation rows": len(somatic_mutations),
    "Samples represented by validated somatic mutations": somatic_mutations["Tumor_Sample_Barcode"].nunique(),
    "Mutation records linked to clinical samples (%)": round(
        somatic_mutations["Tumor_Sample_Barcode"].isin(set(sample["SAMPLE_ID"])).mean() * 100, 2
    ),
})
integrity.to_frame("value")

## 3. Missingness

Clinical cancer datasets commonly contain variables that are not applicable or were not collected uniformly. Missingness is described before any modelling is considered.

In [ ]:
def missingness_summary(df):
    return (
        pd.DataFrame({
            "missing_count": df.isna().sum(),
            "missing_percent": df.isna().mean().mul(100).round(1),
        })
        .sort_values(["missing_percent", "missing_count"], ascending=False)
    )

patient_missingness = missingness_summary(patient)
sample_missingness = missingness_summary(sample)
patient_missingness.head(12)

The `AGE` and `SEX` fields in this cBioPortal export are entirely missing, so they must not be used in downstream analysis. This is an important data-quality finding rather than a value to impute.

In [ ]:
clinical_snapshot = pd.Series({
    "Patients": patient["PATIENT_ID"].nunique(),
    "Primary samples": (sample["SAMPLE_TYPE"] == "Primary").sum(),
    "Living at last follow-up": (patient["OS_STATUS"] == "0:LIVING").sum(),
    "Deceased at last follow-up": (patient["OS_STATUS"] == "1:DECEASED").sum(),
    "Missing overall-survival months": patient["OS_MONTHS"].isna().sum(),
})
clinical_snapshot.to_frame("value")

## 4. Most frequently mutated genes

Frequency is calculated as the number of **unique tumor samples** with at least one reported mutation in a gene. Counting unique samples avoids inflating frequency when one sample has multiple variants in the same gene.

In [ ]:
profiled_samples = sample["SAMPLE_ID"].nunique()

top_genes = (
    somatic_mutations.groupby("Hugo_Symbol")["Tumor_Sample_Barcode"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
    .rename("altered_samples")
    .reset_index()
    .rename(columns={"Hugo_Symbol": "gene"})
)
top_genes["frequency_percent"] = (top_genes["altered_samples"] / profiled_samples * 100).round(1)
top_genes.to_csv(TABLE_DIR / "top_15_mutated_genes.csv", index=False)
top_genes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_data = top_genes.sort_values("frequency_percent")
sns.barplot(data=plot_data, x="frequency_percent", y="gene", color="#176B87", ax=ax)
ax.set(title="Most frequently mutated genes in TCGA-LAML", xlabel="Samples with mutation (%)", ylabel="")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3, fontsize=8)
ax.set_xlim(0, max(plot_data["frequency_percent"]) * 1.18)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "top_15_mutated_genes.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Variant classification landscape

The following chart summarizes the reported functional classifications. It describes the dataset and should not be interpreted as evidence that every reported variant is a clinically actionable driver.

In [ ]:
variant_classes = (
    somatic_mutations["Variant_Classification"]
    .fillna("Missing")
    .value_counts()
    .head(12)
    .rename_axis("variant_classification")
    .reset_index(name="variant_count")
)
variant_classes.to_csv(TABLE_DIR / "variant_classification_counts.csv", index=False)
variant_classes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_data = variant_classes.sort_values("variant_count")
sns.barplot(data=plot_data, x="variant_count", y="variant_classification", color="#2A9D8F", ax=ax)
ax.set(title="Most common variant classifications", xlabel="Reported variants", ylabel="")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "variant_classification_counts.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Per-sample reported mutation burden

This count is a descriptive measure of rows in the mutation table, not a standardized clinical tumor mutational burden. Extreme values may reflect true biology, sequencing, processing, or annotation differences and require investigation.

In [ ]:
sample_mutation_counts = (
    somatic_mutations.groupby("Tumor_Sample_Barcode").size().rename("reported_mutations")
    .reindex(sample["SAMPLE_ID"], fill_value=0)
)
sample_mutation_counts.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame().T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(sample_mutation_counts.clip(upper=sample_mutation_counts.quantile(0.95)), bins=25, color="#E76F51", ax=ax)
ax.set(title="Reported mutations per sample (values capped at 95th percentile for display)", xlabel="Reported mutation rows", ylabel="Number of samples")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "reported_mutations_per_sample.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Conclusions and limitations

- The clinical and sample tables contain 200 unique patients and 200 primary samples.
- Mutation records link successfully to the clinical sample table.
- Key demographic fields such as age and sex are unavailable in this export and will not be imputed.
- Mutation prevalence is measured by unique altered samples, not raw mutation-row counts.
- Survival analysis is intentionally deferred until censoring, missingness, endpoint definitions, and modelling assumptions are handled correctly.
- The cohort is historical and dataset-specific; associations are exploratory and require external validation.
- No raw patient-level data or participant identifiers should be committed to the public repository.

### Data citation

The Cancer Genome Atlas (TCGA), Acute Myeloid Leukemia PanCancer Atlas cohort, accessed through cBioPortal. Study identifier: `laml_tcga_pan_can_atlas_2018`. Accessed September 1, 2026. See the repository README for source links and publication citations.